# V18: MDS on Recipe Vectors vs Free Sorting Reference

Applies the same MDS + threshold clustering pipeline from V17 (free sorting) to the **ingredient-based recipe vectors** from V9.  
This answers: *does our threshold-weighted feature space produce a map that resembles the panelist free sorting result?*

**Models compared:**
- M1: OT1 Grandfamilien (no threshold)
- M2: OT1 Grandfamilien + threshold weighting  
- M3: OT1+OT2+OT3 weighted (no threshold)
- M4: OT1+OT2+OT3 weighted + threshold

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.manifold import MDS
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLUSTER_COLORS = [
    '#E63946', '#457B9D', '#2A9D8F', '#E9C46A',
    '#F4A261', '#A8DADC', '#6A0572', '#264653',
]

print('Libraries loaded.')

## 1. Load Data (identical to V9)

In [ ]:
DATA_PATH   = '../data/gold/Versuchsdaten_3_1.csv'
IGNORE_PATH = '../data/gold/ignone_substances.csv'
CAS_PATH    = '../data/gold/CAS Nummern.csv'

df_raw = pd.read_csv(DATA_PATH)
ign    = pd.read_csv(IGNORE_PATH)
cas    = pd.read_csv(CAS_PATH, header=13)

# Resolve ignore list
ign_cas = ign[['Ident']].merge(
    cas[['Ident.', 'CAS-Nr.: - Hinweis 1']].rename(columns={'Ident.': 'Ident'}),
    on='Ident', how='left'
)
cas_to_ignore   = set(ign_cas['CAS-Nr.: - Hinweis 1'].dropna().astype(str).str.strip())
names_to_ignore = {str(n).lower().strip() for n in ign['Name']}

df = df_raw.copy()
df['_cas'] = df['CAS-Nr.: - Hinweis 1'].astype(str).str.strip()
df.loc[df['_cas'].isin(cas_to_ignore), 'Totalmenge'] = 0.0
df.drop(columns='_cas', inplace=True)
df.loc[df['Name'].str.lower().str.strip().isin(names_to_ignore), 'Totalmenge'] = 0.0

per_recipe_total = df.groupby('Rez.-Nr.')['Totalmenge'].transform('sum')
df['Totalmenge'] = np.where(per_recipe_total > 0,
                            df['Totalmenge'] / per_recipe_total,
                            df['Totalmenge'])

recipes = df['Rez.-Nr.'].unique().tolist()
print(f'Recipes: {len(recipes)}')
print(f'Sample: {recipes[:5]}')

## 2. Feature Helpers (identical to V9)

In [ ]:
OT1 = 'Odour Type 1 FlavourWheel'
OT2 = 'Odour Type 2 Flavour Wheel'
OT3 = 'Odour Type 3 Flavour Wheel'
THRESHOLD_COL = 'Threshold ppm (Datenbank)'

def pos_weight(position, n_cols):
    return (n_cols + 1 - position) / n_cols

def thresh_factor(threshold_ppm, fallback=1.0):
    try:
        raw = str(threshold_ppm).strip().replace(',', '.')
        t = float(raw)
        return 1.0 / t if (not np.isnan(t) and t > 0) else fallback
    except (TypeError, ValueError):
        return fallback

def norm_term(term):
    if pd.isna(term) or not isinstance(term, str):
        return None
    t = term.lower().strip().replace('"', '').replace("'", '').rstrip('.,;:')
    return t if len(t) >= 2 else None

def build_vocabulary(df, feature_cols):
    all_terms = set()
    for col in feature_cols:
        if col in df.columns:
            for t in df[col].dropna().map(norm_term):
                if t:
                    all_terms.add(t)
    return sorted(all_terms)

def build_recipe_vectors(df, recipes, feature_cols_weighted, use_threshold):
    feature_cols = [col for col, _ in feature_cols_weighted]
    vocab = build_vocabulary(df, feature_cols)
    vocab_to_idx = {t: i for i, t in enumerate(vocab)}
    n_recipes = len(recipes)
    n_feat    = len(vocab)
    vectors   = np.zeros((n_recipes, n_feat), dtype=np.float64)
    for r_idx, recipe in enumerate(recipes):
        rows = df[df['Rez.-Nr.'] == recipe]
        for _, row in rows.iterrows():
            qty = float(row['Totalmenge'])
            if qty <= 0:
                continue
            t_fac     = thresh_factor(row[THRESHOLD_COL]) if use_threshold else 1.0
            ingr_base = qty * t_fac
            for col, col_weight in feature_cols_weighted:
                if col not in df.columns:
                    continue
                term = norm_term(row.get(col))
                if term and term in vocab_to_idx:
                    vectors[r_idx, vocab_to_idx[term]] += col_weight * ingr_base
    return vocab, normalize(vectors)

print('Helpers defined.')

## 3. MDS Pipeline

For each model:
1. Build normalised recipe vectors (same as V9)
2. Compute **cosine dissimilarity** matrix between all 25 recipes
3. Apply **metric MDS** → 2D coordinates
4. Apply **Ward + threshold** → cluster assignments
5. Plot with confidence ellipses (same style as free sorting reference)

In [ ]:
def cosine_dissimilarity(vecs):
    """Pairwise cosine dissimilarity: 1 - cosine_similarity."""
    # vectors are already L2-normalised, so dot product = cosine similarity
    sim = vecs @ vecs.T
    sim = np.clip(sim, -1.0, 1.0)
    diss = 1.0 - sim
    np.fill_diagonal(diss, 0.0)
    return diss

def confidence_ellipse(x, y, ax, n_std=1.5, **kwargs):
    if len(x) < 2:
        ax.scatter(x, y, s=80, color=kwargs.get('facecolor', 'gray'), zorder=4)
        return
    cov = np.cov(x, y)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width  = max(2 * n_std * np.sqrt(abs(eigenvalues[0])), 0.001)
    height = max(2 * n_std * np.sqrt(abs(eigenvalues[1])), 0.001)
    ellipse = Ellipse(xy=(np.mean(x), np.mean(y)),
                      width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)

def mds_plot(ax, coords, names, cluster_labels, threshold, title):
    n_c = len(np.unique(cluster_labels))
    colors = CLUSTER_COLORS[:n_c]
    ax.set_facecolor('#FAFAFA')
    ax.axhline(0, color='#CCCCCC', lw=0.7, zorder=1)
    ax.axvline(0, color='#CCCCCC', lw=0.7, zorder=1)
    for c_idx, cid in enumerate(sorted(np.unique(cluster_labels))):
        mask = cluster_labels == cid
        cx, cy = coords[mask, 0], coords[mask, 1]
        col = colors[c_idx % len(colors)]
        confidence_ellipse(cx, cy, ax, n_std=1.5,
                           facecolor=col, alpha=0.15,
                           edgecolor=col, linewidth=1.3, linestyle='--', zorder=2)
        ax.scatter(cx, cy, color=col, s=60, zorder=4, edgecolors='white', lw=0.7)
        for i, name in enumerate(np.array(names)[mask]):
            ax.annotate(name, (cx[i], cy[i]), fontsize=6.5, ha='center', va='bottom',
                        xytext=(0, 4), textcoords='offset points', color=col)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('MDS Dim 1', fontsize=8)
    ax.set_ylabel('MDS Dim 2', fontsize=8)
    ax.grid(True, alpha=0.2, lw=0.4)
    ax.tick_params(labelsize=7)

print('Pipeline helpers defined.')

In [ ]:
MODEL_CONFIGS = [
    {
        'name': 'M1: OT1 Grandfamilien\n(no threshold)',
        'feature_cols_weighted': [(OT1, 1.0)],
        'use_threshold': False,
    },
    {
        'name': 'M2: OT1 Grandfamilien\n(+ threshold weighting)',
        'feature_cols_weighted': [(OT1, 1.0)],
        'use_threshold': True,
    },
    {
        'name': 'M3: OT1+OT2+OT3 weighted\n(no threshold)',
        'feature_cols_weighted': [
            (OT1, pos_weight(1, 4)),
            (OT2, pos_weight(2, 4)),
            (OT3, pos_weight(3, 4)),
        ],
        'use_threshold': False,
    },
    {
        'name': 'M4: OT1+OT2+OT3 weighted\n(+ threshold weighting)',
        'feature_cols_weighted': [
            (OT1, pos_weight(1, 4)),
            (OT2, pos_weight(2, 4)),
            (OT3, pos_weight(3, 4)),
        ],
        'use_threshold': True,
    },
]

# Threshold for cluster cuts — adjust to taste
THRESHOLD = 0.3

all_results = []
for cfg in MODEL_CONFIGS:
    vocab, vecs = build_recipe_vectors(
        df, recipes,
        feature_cols_weighted=cfg['feature_cols_weighted'],
        use_threshold=cfg['use_threshold'],
    )
    diss = cosine_dissimilarity(vecs)
    
    mds = MDS(n_components=2, dissimilarity='precomputed',
              metric=True, n_init=10, max_iter=1000,
              random_state=42, normalized_stress='auto')
    coords = mds.fit_transform(diss)
    
    condensed = squareform(diss, checks=False)
    Z = linkage(condensed, method='ward')
    labels = fcluster(Z, t=THRESHOLD, criterion='distance')
    n_c = len(np.unique(labels))
    
    all_results.append({
        'name': cfg['name'],
        'coords': coords,
        'labels': labels,
        'stress': mds.stress_,
        'n_clusters': n_c,
        'Z': Z,
    })
    print(f"{cfg['name'].split(chr(10))[0]}  →  stress={mds.stress_:.4f}, {n_c} clusters")

## 4. Threshold Scan
Find a threshold that gives a meaningful number of clusters (comparable to free sorting result).

In [ ]:
# Use M2 (OT1 + threshold) as the reference model for threshold scanning
vocab_m2, vecs_m2 = build_recipe_vectors(
    df, recipes,
    feature_cols_weighted=[(OT1, 1.0)],
    use_threshold=True,
)
diss_m2 = cosine_dissimilarity(vecs_m2)
Z_m2 = linkage(squareform(diss_m2, checks=False), method='ward')

print('Threshold scan for M2 (OT1 + threshold):')
for t in np.arange(0.05, 0.55, 0.05):
    lbl = fcluster(Z_m2, t=t, criterion='distance')
    n_c = len(np.unique(lbl))
    print(f'  threshold={t:.2f}  →  {n_c} clusters')

## 5. Main Comparison: 4 Models Side by Side

In [ ]:
# Re-run with a threshold chosen from the scan above
THRESHOLD = 0.25  # adjust based on scan output

all_results = []
for cfg in MODEL_CONFIGS:
    vocab, vecs = build_recipe_vectors(
        df, recipes,
        feature_cols_weighted=cfg['feature_cols_weighted'],
        use_threshold=cfg['use_threshold'],
    )
    diss = cosine_dissimilarity(vecs)
    mds = MDS(n_components=2, dissimilarity='precomputed',
              metric=True, n_init=10, max_iter=1000,
              random_state=42, normalized_stress='auto')
    coords = mds.fit_transform(diss)
    Z = linkage(squareform(diss, checks=False), method='ward')
    labels = fcluster(Z, t=THRESHOLD, criterion='distance')
    all_results.append({
        'name': cfg['name'],
        'coords': coords, 'labels': labels,
        'stress': mds.stress_, 'n_clusters': len(np.unique(labels)),
    })

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, res in zip(axes, all_results):
    n_c = res['n_clusters']
    title = f"{res['name']}\nthreshold={THRESHOLD:.2f}, {n_c} clusters, stress={res['stress']:.3f}"
    mds_plot(ax, res['coords'], recipes, res['labels'], THRESHOLD, title)

fig.suptitle(
    'MDS Consensus Map — 4 Models (cosine dissimilarity + Ward threshold clustering)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v18_mds_4models.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

## 6. Direct Comparison: Free Sorting vs Best Threshold Model

Side-by-side: V17 free sorting MDS (panelist data) vs M2 threshold MDS (ingredient data).  
Products that appear together in both maps are truly similar — both by smell perception AND by ingredient composition.

In [ ]:
import openpyxl
from pathlib import Path

# Load free sorting XLStat STATIS coordinates
EXCEL_PATH = Path('../free_sorting/Auswertung Best Of XLStat Erdbeere.xlsm')
wb = openpyxl.load_workbook(EXCEL_PATH, keep_vba=True, data_only=True)

# Load free sorting raw data for dissimilarity + clustering
ws_data = wb['Tabelle1 (7)']
rows = list(ws_data.iter_rows(min_row=1, max_row=26, values_only=True))
fs_products = [r[0] for r in rows[1:] if r[0] is not None]
fs_matrix = np.array([[r[i] for i in range(1, 13)] for r in rows[1:] if r[0] is not None], dtype=float)

n_fs = len(fs_products)
co_occ = np.zeros((n_fs, n_fs))
for a in range(12):
    g = fs_matrix[:, a]
    for i in range(n_fs):
        for j in range(n_fs):
            if g[i] == g[j]:
                co_occ[i, j] += 1
fs_diss = 1.0 - co_occ / 12
np.fill_diagonal(fs_diss, 0.0)

# Load XLStat STATIS coordinates (rows 132-156 in Free Sorting7)
ws_xlstat = wb['Free Sorting7']
xlstat_rows = list(ws_xlstat.iter_rows(min_row=132, max_row=157, values_only=True))
xlstat_products, xlstat_d1, xlstat_d2 = [], [], []
for row in xlstat_rows:
    vals = [v for v in row if v is not None]
    if vals and isinstance(vals[0], str) and vals[0].strip() and isinstance(vals[1], (int, float)):
        xlstat_products.append(vals[0])
        xlstat_d1.append(float(vals[1]))
        xlstat_d2.append(float(vals[2]))
xlstat_coords = np.column_stack([xlstat_d1, xlstat_d2])

# Ward clustering on free sorting dissimilarity
FS_THRESHOLD = 1.2
Z_fs = linkage(squareform(fs_diss, checks=False), method='ward')
fs_labels = fcluster(Z_fs, t=FS_THRESHOLD, criterion='distance')
fs_labels_xlstat = np.array([fs_labels[fs_products.index(p)] for p in xlstat_products])

print(f'Free sorting: {len(np.unique(fs_labels))} clusters at threshold={FS_THRESHOLD}')
print(f'Ingredient products: {len(recipes)}')
print(f'Free sorting products: {len(fs_products)}')

In [ ]:
# M2 (OT1 + threshold) — best model for direct comparison
vocab_m2, vecs_m2 = build_recipe_vectors(
    df, recipes,
    feature_cols_weighted=[(OT1, 1.0)],
    use_threshold=True,
)
diss_m2 = cosine_dissimilarity(vecs_m2)
mds_m2 = MDS(n_components=2, dissimilarity='precomputed',
             metric=True, n_init=10, max_iter=1000,
             random_state=42, normalized_stress='auto')
coords_m2 = mds_m2.fit_transform(diss_m2)
Z_m2 = linkage(squareform(diss_m2, checks=False), method='ward')

ING_THRESHOLD = 0.25  # adjust from scan
labels_m2 = fcluster(Z_m2, t=ING_THRESHOLD, criterion='distance')
n_c_m2 = len(np.unique(labels_m2))

print(f'M2 ingredient MDS: stress={mds_m2.stress_:.4f}, {n_c_m2} clusters')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Left: Free Sorting (XLStat coords)
mds_plot(ax1, xlstat_coords, xlstat_products, fs_labels_xlstat,
         FS_THRESHOLD,
         f'Free Sorting (XLStat STATIS)\nthreshold={FS_THRESHOLD}, {len(np.unique(fs_labels_xlstat))} clusters')
ax1.set_xlabel('Dim 1  (XLStat STATIS)', fontsize=9)
ax1.set_ylabel('Dim 2  (XLStat STATIS)', fontsize=9)

# Right: Ingredient MDS (threshold model)
mds_plot(ax2, coords_m2, recipes, labels_m2,
         ING_THRESHOLD,
         f'Ingredient MDS — M2: OT1 + threshold\nthreshold={ING_THRESHOLD}, {n_c_m2} clusters, stress={mds_m2.stress_:.3f}')

fig.suptitle(
    'Panelist Free Sorting vs Ingredient-Based MDS\n'
    'Products clustered similarly in both maps = perceptual + compositional alignment',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v18_free_sorting_vs_ingredient_mds.png', dpi=150, bbox_inches='tight')
plt.show()
print('Side-by-side comparison saved.')

## 7. Key Findings

### What this compares
| Map | Data source | Distance metric |
|-----|------------|----------------|
| Free Sorting (left) | Panelist groupings | Co-occurrence dissimilarity |
| Ingredient MDS (right) | Odour type composition × threshold | Cosine dissimilarity |

### How to read the comparison
- **Recipes that cluster together in both maps** → strong alignment between what panelists perceive and what the ingredient composition predicts
- **Recipes that cluster differently** → perception diverges from composition (interesting outliers for investigation)
- The **threshold weighting** (1/threshold_ppm) upweights ingredients with low odour detection thresholds — those that smell strongly even in small amounts

### Plots produced
- `v18_mds_4models.png` — all 4 models (with/without threshold × OT1 / OT1+OT2+OT3)
- `v18_free_sorting_vs_ingredient_mds.png` — direct side-by-side comparison